In [1]:
import pandas as pd
import numpy as np

from gensim.models import Word2Vec

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer

from utils.logger import Logger
from w2v_model import KmerGenerator

import os, pickle

In [2]:
logger = Logger("./logs/")

In [3]:
class w2vTransformer(BaseEstimator, TransformerMixin):
    """Scikit-learn compatible transformer that uses pre-trained Word2Vec models"""
    
    def __init__(self, model_path, k=3):
        self.model_path = model_path
        self.k = k
        self.model = None
        self.kmer_gen = KmerGenerator(k=k)
        self.vector_size = None
    
    def fit(self, X, y=None):
        """Load the pre-trained Word2Vec model"""
        logger.log_info(f"Loading pre-trained Word2Vec model from {self.model_path}")
        
        if not os.path.exists(self.model_path):
            raise FileNotFoundError(f"Pre-trained Word2Vec model not found at {self.model_path}")
        
        self.model = Word2Vec.load(self.model_path)
        self.vector_size = self.model.wv.vector_size
        
        logger.log_info(f"Loaded Word2Vec model with vocabulary size: {len(self.model.wv.key_to_index)}")
        logger.log_info(f"Vector size: {self.vector_size}")
        
        return self
    
    def transform(self, X):
        """Transform sequences to embeddings using pre-trained model"""
        if self.model is None:
            raise ValueError("Model not loaded yet. Call fit() first.")
        
        # Convert to list if it's a pandas Series
        sequences = X.tolist() if hasattr(X, 'tolist') else list(X)
        
        embeddings = []
        for seq in sequences:
            kmers = self.kmer_gen.generate_kmers(str(seq).upper())
            
            # Get embeddings for each k-mer
            kmer_embeddings = []
            for kmer in kmers:
                if kmer in self.model.wv.key_to_index:
                    kmer_embeddings.append(self.model.wv[kmer])
                else:
                    # Use zero vector for unknown k-mers
                    kmer_embeddings.append(np.zeros(self.vector_size))
            
            if kmer_embeddings:
                # Average the k-mer embeddings
                seq_embedding = np.mean(kmer_embeddings, axis=0)
            else:
                seq_embedding = np.zeros(self.vector_size)
            
            embeddings.append(seq_embedding)
        
        return np.array(embeddings)

In [4]:
class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    """Scikit-learn compatible Label Encoder transformer"""
    
    def __init__(self):
        self.label_encoder = LabelEncoder()
    
    def fit(self, X, y=None):
        # Convert to list if it's a pandas Series
        X_list = X.tolist() if hasattr(X, 'tolist') else list(X)
        self.label_encoder.fit([str(x) for x in X_list])
        return self
    
    def transform(self, X):
        # Convert to list if it's a pandas Series
        X_list = X.tolist() if hasattr(X, 'tolist') else list(X)
        encoded = self.label_encoder.transform([str(x) for x in X_list])
        return encoded.reshape(-1, 1)

In [5]:
class ColumnDropperTransformer(BaseEstimator, TransformerMixin):
    """Transformer to drop specified columns"""
    
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.columns_to_drop, errors='ignore')
        else:
            # If it's not a DataFrame, return as is
            return X

In [ ]:
class GenomicPreprocessor:
    """Complete preprocessing pipeline for genomic data using pre-trained Word2Vec models"""
    
    def __init__(self, ref_model_path, alt_model_path, k=3):
        self.ref_model_path = ref_model_path
        self.alt_model_path = alt_model_path
        self.k = k
        self.pipeline = None
        self.feature_names = None
        self.vector_size = None
        
    def build_pipeline(self):
        """Build the preprocessing pipeline with pre-trained models"""
        
        # Create transformers using pre-trained models
        ref_transformer = Pipeline([
            ('word2vec', w2vTransformer(
                model_path=self.ref_model_path,
                k=self.k
            ))
        ])
        
        alt_transformer = Pipeline([
            ('word2vec', w2vTransformer(
                model_path=self.alt_model_path,
                k=self.k
            ))
        ])
        
        chr_transformer = Pipeline([
            ('label_encoder', LabelEncoderTransformer())
        ])
        
        # Create column transformer
        preprocessor = ColumnTransformer([
            ('ref', ref_transformer, ['ref']),
            ('alt', alt_transformer, ['alt']),
            ('chr', chr_transformer, ['chr']),
            ('passthrough', 'passthrough', None)  # Will be set dynamically
        ])
        
        # Complete pipeline
        self.pipeline = Pipeline([
            ('drop_gene', ColumnDropperTransformer(['gene'])),
            ('preprocessor', preprocessor)
        ])
        
        return self.pipeline
    
    def fit(self, X, y=None):
        """Fit the preprocessing pipeline"""
        if self.pipeline is None:
            self.build_pipeline()
        
        # Identify numeric columns (exclude categorical and target)
        categorical_cols = ['chr', 'ref', 'alt', 'gene']
        target_col = ['class']
        numeric_cols = [col for col in X.columns 
                       if col not in categorical_cols + target_col]
        
        # Update the passthrough columns in the pipeline
        self.pipeline.named_steps['preprocessor'].transformers[3] = (
            'passthrough', 'passthrough', numeric_cols
        )
        
        logger.log_info(f"Fitting preprocessing pipeline on {X.shape[0]} samples...")
        self.pipeline.fit(X)
        
        # Get vector size from the fitted transformer
        ref_transformer = self.pipeline.named_steps['preprocessor'].named_transformers_['ref']
        self.vector_size = ref_transformer.named_steps['word2vec'].vector_size
        
        # Generate feature names
        self._generate_feature_names(numeric_cols)
        
        return self
    
    def transform(self, X):
        """Transform data using the fitted pipeline"""
        if self.pipeline is None:
            raise ValueError("Pipeline not fitted yet. Call fit() first.")
        
        logger.log_info(f"Transforming {X.shape[0]} samples...")
        transformed = self.pipeline.transform(X)
        
        return transformed
    
    def fit_transform(self, X, y=None):
        """Fit and transform in one step"""
        return self.fit(X, y).transform(X)
    
    def _generate_feature_names(self, numeric_cols):
        """Generate feature names for the transformed data"""
        feature_names = []
        
        # Ref embeddings
        feature_names.extend([f'ref_emb_{i}' for i in range(self.vector_size)])
        
        # Alt embeddings
        feature_names.extend([f'alt_emb_{i}' for i in range(self.vector_size)])
        
        # Chr encoded
        feature_names.append('chr_encoded')
        
        # Numeric columns
        feature_names.extend(numeric_cols)
        
        self.feature_names = feature_names
    
    def get_feature_names(self):
        """Get feature names after transformation"""
        return self.feature_names
    
    def save_pipeline(self, filepath):
        """Save the fitted pipeline"""
        with open(filepath, 'wb') as f:
            pickle.dump(self.pipeline, f)
        logger.log_info(f"Pipeline saved to {filepath}")
    
    def load_pipeline(self, filepath):
        """Load a fitted pipeline"""
        with open(filepath, 'rb') as f:
            self.pipeline = pickle.load(f)
        logger.log_info(f"Pipeline loaded from {filepath}")

In [8]:
class GenomicDataset(Dataset):
    """PyTorch Dataset for genomic data"""
    
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [9]:
class GenomicClassifier(nn.Module):
    """Neural network for genomic classification"""
    
    def __init__(self, input_size, hidden_sizes=[256, 128, 64], dropout_rate=0.3):
        super(GenomicClassifier, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_size),
                nn.Dropout(dropout_rate)
            ])
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, 2))  # Binary classification
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

In [10]:
def train_model(model, train_loader, val_loader, num_epochs=50, learning_rate=0.001):
    """Train the PyTorch model"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    logger.log_info(f"Training on device: {device}")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch_features, batch_labels in val_loader:
                batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
                
                outputs = model(batch_features)
                loss = criterion(outputs, batch_labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += batch_labels.size(0)
                correct += (predicted == batch_labels).sum().item()
        
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        val_accuracy = 100 * correct / total
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 10 == 0:
            logger.log_info(f'Epoch [{epoch+1}/{num_epochs}], '
                       f'Train Loss: {train_loss:.4f}, '
                       f'Val Loss: {val_loss:.4f}, '
                       f'Val Acc: {val_accuracy:.2f}%')
    
    return train_losses, val_losses, val_accuracies


In [11]:
logger.log_info("Loading data...")
try:
    train_df = pd.read_csv('data/train.csv')
    test_df = pd.read_csv('data/test.csv')
    orthogonal_df = pd.read_csv('data/orthogonal.csv')
    logger.log_info(f"Loaded train dataset with shape: {train_df.shape}")
    logger.log_info(f"Loaded test dataset with shape: {test_df.shape}")
    logger.log_info(f"Loaded orthogonal with shape: {orthogonal_df.shape}")
except FileNotFoundError:
    logger.log_error("train.csv not found. Please ensure the file is in the current directory.")

train_df.columns = list(map(lambda x: x.replace(" ", "_").lower(), train_df.columns))
test_df.columns = list(map(lambda x: x.replace(" ", "_").lower(), test_df.columns))
orthogonal_df.columns = list(map(lambda x: x.replace(" ", "_").lower(), orthogonal_df.columns))

deleted_cols = [col for col in test_df.columns if col not in train_df.columns]
logger.log_info(f"{deleted_cols = }")

test_df_filtered = test_df.drop(columns=deleted_cols)
orthogonal_df_filtered = orthogonal_df.drop(columns=deleted_cols)

logger.log_info("After column selection:")
logger.log_info(f"Train dataset shape: {train_df.shape}")
logger.log_info(f"Test dataset shape: {test_df.shape}")
logger.log_info(f"Orthogonal shape: {orthogonal_df.shape}")


required_cols = ['chr', 'ref', 'alt', 'class']
missing_cols = [col for col in required_cols if col not in train_df.columns]
if missing_cols:
    logger.log_error(f"Missing required columns: {missing_cols}")

ref_model_path = "./global_models/ref_word2vec.model"
alt_model_path = "./global_models/alt_word2vec.model"

In [12]:
X_train = train_df.drop(['class'], axis=1)
y_train = train_df['class']

X_test = test_df_filtered.drop(['class'], axis=1)
y_test = test_df_filtered['class']

X_orthogonal = orthogonal_df_filtered.drop(['class'], axis=1)
y_orthogonal = orthogonal_df_filtered['class']

In [13]:
logger.log_info("Step 2: Creating preprocessing pipeline with pre-trained Word2Vec models...")
preprocessor = GenomicPreprocessor(
    ref_model_path=ref_model_path,
    alt_model_path=alt_model_path,
    k=3
)

In [14]:
X_train_transformed = preprocessor.fit_transform(X_train)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 1 and the array at index 3 has size 77002